In [48]:
#install dependencies, you can uncomment if needed
# %pip install scikit-image
# %pip install matplotlib
# %pip install numpy

# print("\nFinished installing scikit, matplotlib, and numpy\n")

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
from skimage.transform import resize
import random
import os
from PIL import Image
import pandas as pd

## Preprocessing: Median filter and Gaussian Filter

In [49]:
import math
import numpy as np

#function for convolution, based on the matlab code showed in class
#This is using zero padding, so the result image is slightly larger than the original image.

def twoD_convolution(image, kernel):
    image_height = image.shape[0]
    image_width = image.shape[1]

    kernel_height = kernel.shape[0]
    kernel_width = kernel.shape[1]
    result_height = image_height + kernel_height - 1
    result_width = image_width + kernel_width - 1
    
    # Initialize result with zeros
    result = np.zeros((result_height, result_width))
    for i in range(image_height):
        for j in range(image_width):
            # For each pixel in the image, multiply it by the entire kernel
            # and add it to the corresponding neighborhood in the result
            result[i : i + kernel_height, j : j + kernel_width] += image[i, j] * kernel

    pad_h = kernel_height // 2
    pad_w = kernel_width // 2
    return result[pad_h : pad_h + image_height, pad_w : pad_w + image_width]

def median_filter(img, kernel_size):
    pad = kernel_size // 2
    #pad the image with zeros on the edges.
    padded = np.pad(img, pad, mode='constant', constant_values=0)

    #initialize the result matrix with zeros (same size as original image)
    result = np.zeros_like(img)
    
    #replace each pixel with the median of the neighboring pixels (inside the kernel)
    #this loops through the entire image
    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            region = padded[i:i+kernel_size, j:j+kernel_size]
            result[i, j] = np.median(region)
    
    return result

def gaussian_filter_equation(x,y,sigma):
    return math.exp(-(x**2 + y**2) / (2 * sigma**2))

def gaussian_filter(img, kernel_size, sigma):
    k = kernel_size // 2
    kernel = np.zeros((kernel_size, kernel_size))
    for i in range(kernel_size):
        for j in range(kernel_size):
            kernel[i, j] = gaussian_filter_equation(i - k, j - k, sigma)
    kernel /= kernel.sum()
    return twoD_convolution(img, kernel)


### Reading in the images, converting them to grayscale using the Luminosity method, and using the filters.

In [50]:
from matplotlib.image import imread
from PIL import Image
def luminosity(image):
    r = 0.21
    g = 0.72
    b = 0.07
    img = image.copy()
    #create 3 copies of the original image.
    red = img[:, :, 0]
    green = img[:, :, 1]
    blue = img[:, :, 2]

    return (red* r) + (green * g) + (blue * b)

if not os.path.exists("median_filtered_images"):
    os.makedirs("median_filtered_images")
    print(f"Created folder: median_filtered_images")
else:
    print(f"Folder median_filtered_images already exists.")

# There are 10 dataset images, all named the same way (img_1, img_2, img_3...)
print("Median Filtering 10 Images\n")
for i in range(1, 11):
    print("Filtering Image " + str(i))
    #luminsoity -> median filter -> save the result to median_filtered_images folder
    current_image = luminosity(imread("dataset_images/img_" + str(i) + ".jpg"))
    median_filtered = median_filter(current_image, 3) #kernel size of 3
    res_img = Image.fromarray(median_filtered.astype(np.uint8))
    res_img.convert("L").save(f"median_filtered_images/median_img_{i}.jpg")
    # plt.imsave("median_filtered_images/median_img_" + str(i) + ".jpg", median_filtered, cmap='gray')




Folder median_filtered_images already exists.
Median Filtering 10 Images

Filtering Image 1
Filtering Image 2
Filtering Image 3
Filtering Image 4
Filtering Image 5
Filtering Image 6
Filtering Image 7
Filtering Image 8
Filtering Image 9
Filtering Image 10


In [51]:
if not os.path.exists("median_gaussian_filtered_images"):
    os.makedirs("median_gaussian_filtered_images")
    print(f"Created folder: median_gaussian_filtered_images")
else:
    print(f"Folder median_gaussian_filtered_images already exists.")
    
print("Gaussian Filtering 10 Images\n")
for i in range(1, 11):
    print("Filtering Image " + str(i))
    #gaussian filter -> save the result to median_gaussian_filtered_images folder
    current_image = imread("median_filtered_images/median_img_" + str(i) + ".jpg")
    gaussian_filtered = gaussian_filter(current_image, 3, 0.5) #kernel size of 3 and sigma of 0.5
    res_img = Image.fromarray(gaussian_filtered.astype(np.uint8))
    res_img.convert("L").save(f"median_gaussian_filtered_images/median_gaussian_img_{i}.jpg")

Folder median_gaussian_filtered_images already exists.
Gaussian Filtering 10 Images

Filtering Image 1
Filtering Image 2
Filtering Image 3
Filtering Image 4
Filtering Image 5
Filtering Image 6
Filtering Image 7
Filtering Image 8
Filtering Image 9
Filtering Image 10


### Define the piecewise contrast stretching function:

In [52]:
def piecewise_contrast(img, t):
    image = img.copy()
    image = image.astype(np.float64)
    L_min = 0
    L_max = 255
    i_min = np.min(image)
    i_max = np.max(image)

    result = np.zeros_like(image)
    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            if image[i, j] <= t:
                result[i, j] = ((image[i,j] - i_min) * (t - L_min) / (t - i_min + 0.1)) + L_min
            else:
                result[i, j] = ((image[i,j] - t + 1) *(L_max - t+1) / (i_max - t+1)) + t + 1
    return result

### Part A: EME (Contrast Ratio Only)
$$EME = \frac{1}{k_1 k_2} \sum_{k=1}^{k_1} \sum_{l=1}^{k_2} \left( \frac{I_{max,k,l}}{I_{min,k,l} + c} \right)$$

In [60]:
# takes in image, M and N (aka k1 and k2).
#M = vertical blocks, N = horizontal
def get_eme(img, vertical_blocks, horizontal_blocks):
    rows, cols = img.shape
    image = (img.copy()).astype(float)  #I don't want this function to modify the image passed in, so I make a copy and convert that to float.
   
    # floor division to get the number of rows and columns in each block.
    block_rows = rows//vertical_blocks
    block_cols = cols//horizontal_blocks

    eme = 0.0
    for r in range(vertical_blocks):
        for c in range(horizontal_blocks):
           row_start = r*block_rows
           row_end = row_start + block_rows + 1
           col_start = c*block_cols
           col_end = col_start + block_cols + 1

           block = image[row_start:row_end, col_start:col_end]  #Extract the current block of the image.
           block_max_intensity = np.max(block)  
           block_min_intensity = np.min(block) 

           block_min_intensity += 0.1   #(the denomimator, c= 0.1 so we add it) 

           eme += (block_max_intensity / block_min_intensity)  #Apply the EME formula and accumulate the result.    
            
    return float(eme) / (vertical_blocks * horizontal_blocks)  #Average the EME value by dividing by the total number of blocks. (1/MN)

part_a_results = []  #will contain all t values and their EME values, including optimal t. For every image. 
for i in range(1, 11):
    print("Processing image " + str(i))
    current_image_max_eme = -1
    current_image_t_optimal = -1
    t_eme_pairs = {}
    img_name = f"median_gaussian_filtered_images/median_gaussian_img_{i}.jpg"
    for t in range(0, 256): #loop through all possible t values (0-255)
        current_image = imread(img_name)
        if(np.max(current_image) <= 1.0):
            current_image = current_image * 255.0
        contrast_stretched_image = piecewise_contrast(current_image, t)
        eme = get_eme(contrast_stretched_image, 8, 8) #use 8x8 blocks
        
        if eme > current_image_max_eme:
            current_image_max_eme = eme
            current_image_t_optimal = t

        t_eme_pairs[t] = eme
    part_a_results.append({
            "image_#": i,
            "t_optimal": current_image_t_optimal,
            "max_eme": current_image_max_eme,
            "all_values": t_eme_pairs  # for later plotting of t and eme
        })    

#to show what the results look like.
#It's pretty long so I won't print the results for every image
print(part_a_results[0])

Processing image 1


Processing image 2
Processing image 3
Processing image 4
Processing image 5
Processing image 6
Processing image 7
Processing image 8
Processing image 9
Processing image 10
{'image_#': 1, 't_optimal': 24, 'max_eme': 580.9509999082095, 'all_values': {0: 579.8324797702624, 1: 579.8324797702624, 2: 580.4020441060482, 3: 580.3852222232019, 4: 580.4385514130256, 5: 580.4621796991023, 6: 580.5806667269658, 7: 580.6099837248914, 8: 580.6618175263959, 9: 580.6588486779573, 10: 580.7193507489119, 11: 580.749629624893, 12: 580.8282261335395, 13: 580.8261584974742, 14: 580.874153343312, 15: 580.9063257976134, 16: 580.904711467635, 17: 580.9186425623668, 18: 580.9173312619121, 19: 580.9331774604422, 20: 580.932076951816, 21: 580.9477843472318, 22: 580.9468397485641, 23: 580.9459772820597, 24: 580.9509999082095, 25: 580.9502606071974, 26: 580.9495781713505, 27: 580.6208789395873, 28: 580.2986934818477, 29: 579.9768637370169, 30: 579.9713790782423, 31: 579.9722965554636, 32: 579.9731569616124, 33: 57

### Part C: EMEE
$$EMEE = \frac{1}{k_1 k_2} \sum_{k=1}^{k_1} \sum_{l=1}^{k_2} \alpha \ln \left( \frac{I_{max,k,l}}{I_{min,k,l} + c} \right)$$

In [ ]:
# takes in image, M and N (aka k1 and k2).
#M = vertical blocks, N = horizontal
def get_emee(img, vertical_blocks, horizontal_blocks, alpha):
    rows, cols = img.shape
    image = (img.copy()).astype(float)  #I don't want this function to modify the image passed in, so I make a copy and convert that to float.
   
    # floor division to get the number of rows and columns in each block.
    block_rows = rows//vertical_blocks
    block_cols = cols//horizontal_blocks

    emee = 0.0
    for r in range(vertical_blocks):
        for c in range(horizontal_blocks):
           row_start = r*block_rows
           row_end = row_start + block_rows + 1
           col_start = c*block_cols
           col_end = col_start + block_cols + 1

           block = image[row_start:row_end, col_start:col_end]  #Extract the current block of the image.
           block_max_intensity = np.max(block)  
           block_min_intensity = np.min(block) 

           block_min_intensity += 0.1   #(the denomimator, c= 0.1 so we add it) 

           emee += alpha * np.log(block_max_intensity / block_min_intensity)  #Apply the EME formula and accumulate the result.    
            
    return float(emee) / (vertical_blocks * horizontal_blocks)  #Average the EME value by dividing by the total number of blocks. (1/MN)

part_c_results = []  #will contain all t values and their EME values, including optimal t. For every image. 
for i in range(1, 11):
    print("Processing image " + str(i))
    current_image_max_eme = -1
    current_image_t_optimal = -1
    t_eme_pairs = {}
    img_name = f"median_gaussian_filtered_images/median_gaussian_img_{i}.jpg"
    for t in range(0, 256): #loop through all possible t values (0-255)
        current_image = imread(img_name)
        if(np.max(current_image) <= 1.0):
            current_image = current_image * 255.0
        contrast_stretched_image = piecewise_contrast(current_image, t)
        eme = get_emee(contrast_stretched_image, 8, 8, 0.5) #use 8x8 blocks and alpha of 0.5
        
        if eme > current_image_max_eme:
            current_image_max_eme = eme
            current_image_t_optimal = t

        t_eme_pairs[t] = eme
    part_a_results.append({
            "image_#": i,
            "t_optimal": current_image_t_optimal,
            "max_eme": current_image_max_eme,
            "all_values": t_eme_pairs  # for later plotting of t and eme
        })    

#to show what the results look like.
#It's pretty long so I won't print the results for every image
print(part_c_results[0])

### Part E: AME
$$AME = \frac{1}{k_1 k_2} \sum_{k=1}^{k_1} \sum_{l=1}^{k_2} \ln \left( \frac{I_{max,k,l} - I_{min,k,l}}{I_{max,k,l} + I_{min,k,l} + c} \right)$$

In [ ]:
# code here